In [4]:
# backtest.py
import pandas as pd
import numpy as np
import itertools
import json

ORDER_SIZE = 5000
CHUNK_SIZE = 100
PARAM_GRID = {
    "lambda_over": [0.1, 1, 10],
    "lambda_under": [0.1, 1, 10],
    "theta_queue": [0.1, 1, 10],
}

class Venue:
    def __init__(self, ask, ask_size, fee=0.001, rebate=0.0):
        self.ask = ask
        self.ask_size = ask_size
        self.fee = fee
        self.rebate = rebate

def compute_cost(split, venues, order_size, lambda_over, lambda_under, theta_queue):
    executed = 0
    cash_spent = 0
    for i in range(len(venues)):
        exe = min(split[i], venues[i].ask_size)
        executed += exe
        cash_spent += exe * (venues[i].ask + venues[i].fee)
        maker_rebate = max(split[i] - exe, 0) * venues[i].rebate
        cash_spent -= maker_rebate

    underfill = max(order_size - executed, 0)
    overfill = max(executed - order_size, 0)
    risk_pen = theta_queue * (underfill + overfill)
    cost_pen = lambda_under * underfill + lambda_over * overfill
    return cash_spent + risk_pen + cost_pen

def allocate(order_size, venues, lambda_over, lambda_under, theta_queue):
    splits = [[]]
    for v in range(len(venues)):
        new_splits = []
        for alloc in splits:
            used = sum(alloc)
            max_v = min(order_size - used, venues[v].ask_size)
            for q in range(0, max_v + 1, CHUNK_SIZE):
                new_splits.append(alloc + [q])
        splits = new_splits

    best_cost = float('inf')
    best_split = []
    for alloc in splits:
        if sum(alloc) != order_size:
            continue
        cost = compute_cost(alloc, venues, order_size, lambda_over, lambda_under, theta_queue)
        if cost < best_cost:
            best_cost = cost
            best_split = alloc
    return best_split, best_cost

def simulate_execution(data, allocator_fn, params):
    remaining = ORDER_SIZE
    total_cost = 0
    total_shares = 0

    for ts, snapshot in data.groupby('ts_event'):
        venues = []
        for _, row in snapshot.iterrows():
            venues.append(Venue(row.ask_px_00, row.ask_sz_00))
        alloc, _ = allocator_fn(remaining, venues, *params)
        cost, filled = 0, 0
        for i, shares in enumerate(alloc):
            exe = min(shares, venues[i].ask_size)
            filled += exe
            cost += exe * venues[i].ask
        remaining -= filled
        total_cost += cost
        total_shares += filled
        if remaining <= 0:
            break

    avg_price = total_cost / total_shares if total_shares > 0 else 0
    return total_cost, avg_price

def best_ask_strategy(data):
    remaining = ORDER_SIZE
    total_cost = 0
    total_shares = 0

    for ts, snapshot in data.groupby('ts_event'):
        best_row = snapshot.loc[snapshot.ask_px_00.idxmin()]
        price = best_row.ask_px_00
        size = best_row.ask_sz_00
        filled = min(remaining, size)
        total_cost += filled * price
        total_shares += filled
        remaining -= filled
        if remaining <= 0:
            break

    avg_price = total_cost / total_shares if total_shares > 0 else 0
    return total_cost, avg_price

def twap_strategy(data):
    bucket = pd.Timedelta("60s")
    start_time = data['ts_event'].min()
    data['bucket'] = ((data['ts_event'] - start_time) // bucket).astype(int)
    buckets = data.groupby('bucket')
    size_per_bucket = ORDER_SIZE / len(buckets)
    remaining = ORDER_SIZE
    total_cost = 0
    total_shares = 0

    for _, group in buckets:
        avg_price = group['ask_px_00'].mean()
        filled = min(size_per_bucket, remaining)
        total_cost += filled * avg_price
        total_shares += filled
        remaining -= filled
        if remaining <= 0:
            break

    avg_price = total_cost / total_shares
    return total_cost, avg_price

def vwap_strategy(data):
    remaining = ORDER_SIZE
    total_cost = 0
    total_shares = 0

    for ts, snapshot in data.groupby('ts_event'):
        weighted_ask = np.average(snapshot.ask_px_00, weights=snapshot.ask_sz_00)
        avg_size = snapshot.ask_sz_00.sum()
        filled = min(remaining, avg_size)
        total_cost += filled * weighted_ask
        total_shares += filled
        remaining -= filled
        if remaining <= 0:
            break

    avg_price = total_cost / total_shares
    return total_cost, avg_price

def load_data():
    path = r"C:\Users\punee\Downloads\l1_day.csv"
    df = pd.read_csv(path, parse_dates=["ts_event"])
    df = df.sort_values(by=["ts_event", "publisher_id"])
    df = df.groupby(["ts_event", "publisher_id"]).head(1)
    return df

def bps_savings(base, optimized):
    return 10_000 * (base - optimized) / base

def main():
    data = load_data()

    best_params = None
    best_cost = float('inf')
    best_price = None

    for l_o, l_u, theta in itertools.product(*PARAM_GRID.values()):
        cost, avg = simulate_execution(data, allocate, (l_o, l_u, theta))
        if cost < best_cost:
            best_cost = cost
            best_params = {"lambda_over": l_o, "lambda_under": l_u, "theta_queue": theta}
            best_price = avg

    baseline_best, avg_best = best_ask_strategy(data)
    baseline_twap, avg_twap = twap_strategy(data)
    baseline_vwap, avg_vwap = vwap_strategy(data)

    print(json.dumps({
        "best_parameters": best_params,
        "optimized": {"total_cost": round(best_cost, 2), "average_price": round(best_price, 4)},
        "best_ask": {"total_cost": round(baseline_best, 2), "average_price": round(avg_best, 4)},
        "twap": {"total_cost": round(baseline_twap, 2), "average_price": round(avg_twap, 4)},
        "vwap": {"total_cost": round(baseline_vwap, 2), "average_price": round(avg_vwap, 4)},
        "savings_vs_best_ask_bps": round(bps_savings(baseline_best, best_cost), 2),
        "savings_vs_twap_bps": round(bps_savings(baseline_twap, best_cost), 2),
        "savings_vs_vwap_bps": round(bps_savings(baseline_vwap, best_cost), 2)
    }, indent=2))

if __name__ == "__main__":
    main()


{
  "best_parameters": {
    "lambda_over": 0.1,
    "lambda_under": 0.1,
    "theta_queue": 0.1
  },
  "optimized": {
    "total_cost": 1113700.0,
    "average_price": 222.74
  },
  "best_ask": {
    "total_cost": 1114102.28,
    "average_price": 222.8205
  },
  "twap": {
    "total_cost": 1115427.41,
    "average_price": 223.0855
  },
  "vwap": {
    "total_cost": 1114102.28,
    "average_price": 222.8205
  },
  "savings_vs_best_ask_bps": 3.61,
  "savings_vs_twap_bps": 15.49,
  "savings_vs_vwap_bps": 3.61
}
